# Milestone 2 Lab, notebook 2 of 6: One calculation, two engines

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the business's records | Recall the question and the answer key, and explore the records profit is calculated from |
| **2. One calculation, two engines** (you are here) | Write a calculation by hand in pandas and in SQL, with test cases that judge both |
| 3. A model drafts a transformation | Ask a model to write the calculation, and let the tests judge it |
| 4. Which model? Accuracy, cost, and speed | Give the same tasks to several models and compare them |
| 5. An agent that tests and fixes its own code | Give a model the tests as a tool, and compare it with a single draft |
| 6. The pipeline, and the same question again | Run the calculations on each engine and ask the analyst the leader's question |

Model requests in this notebook: **0**. A full run of this lab costs well under a dollar.

## What this notebook is for

You write one calculation by hand: **sales** (units sold, gross sales, and discounts for each product and month).

Then you write it a second time in a different language, and see that both give the same answer. That raises a question this notebook answers: if the same rule can be written in pandas, in SQL, or in something else, where does the rule itself live?

There is no AI in this notebook. It builds the yardstick that every AI draft will be measured against.

In [1]:
import pandas as pd
from lab_helpers import load_source_tables, run_sql, show_doc

from retail_profit_stack.transform.contracts import ContractError, conform, tables_differ
from retail_profit_stack.transform.engines import get_engine
from retail_profit_stack.transform.pipeline import load_sources
from retail_profit_stack.transform.specs import SPECS, TASKS
from retail_profit_stack.transform.testing import load_cases, run_cases

tables = load_source_tables()

`retail_profit_stack` is this milestone's production package. In milestone 1 you met it only in the last notebook. Here you will use parts of it from the start, and this notebook shows you what those parts do.

## Step 1: Write the calculation in pandas

Here is the rule for sales, in words:

> For order lines whose order is not cancelled, grouped by the month of the order and by product: add up the quantity, add up quantity times list price, and add up the discount.

In pandas that is four steps: attach each line to its order, drop cancelled orders, work out the month and the line's gross amount, then group and add up.

In [2]:
def sales_pandas(inputs):
    """Units sold, gross sales, and discounts by order month and product."""
    lines = inputs["order_items"].merge(inputs["orders"], on="order_id")
    lines = lines[lines["status"] != "cancelled"]
    lines["month"] = lines["order_date"].dt.strftime("%Y-%m")
    lines["gross"] = lines["quantity"] * lines["unit_list_price"]
    return lines.groupby(["month", "product_id"], as_index=False).agg(
        units_sold=("quantity", "sum"),
        gross_sales=("gross", "sum"),
        discounts=("discount_amount", "sum"),
    )


sales_pandas(tables).head()

,month,product_id,units_sold,gross_sales,discounts
0,2026-09,P001,420,16800.0,840.0
1,2026-09,P002,260,9100.0,455.0
2,2026-09,P003,95,8550.0,855.0
3,2026-09,P004,150,8250.0,0.0
4,2026-09,P005,48,15360.0,768.0


It runs, and the numbers look plausible. **But are they right?** This is the same question you asked of the AI's answer in milestone 1, and the response is the same: you need to know the right answer first.

## Step 2: A test case you can check in your head

We can't check 2,500 lines by hand. So we make up a tiny set of records where we can: two orders, one of them cancelled.

- Order O1, completed: 1 trail water bottle at $40.00
- Order O2, **cancelled**: 3 trail water bottles at $40.00

What should sales be? Only O1 counts. So: 1 unit, $40.00 gross sales, $0.00 discounts.

In [3]:
tiny = {
    "orders": pd.DataFrame({
        "order_id": ["O1", "O2"],
        "order_date": pd.to_datetime(["2026-10-05", "2026-10-06"]),
        "status": ["completed", "cancelled"],
    }),
    "order_items": pd.DataFrame({
        "order_item_id": ["I1", "I2"],
        "order_id": ["O1", "O2"],
        "product_id": ["P001", "P001"],
        "quantity": [1, 3],
        "unit_list_price": [40.0, 40.0],
        "discount_amount": [0.0, 0.0],
    }),
}

sales_pandas(tiny)

,month,product_id,units_sold,gross_sales,discounts
0,2026-10,P001,1,40.0,0.0


One unit, $40.00. Our function agrees with what we worked out by hand.

That is a **hand-worked test case**: a small input, and the output a person worked out. Try breaking the function: delete the line that drops cancelled orders, re-run both cells, and watch the answer become 4 units. The test case catches it. Put the line back afterwards.

## Step 3: Write the same calculation in SQL

SQL is another language for the same job. `run_sql` runs a query over our DataFrames using DuckDB, a database that runs inside Python.

In [5]:
SALES_SQL = """
SELECT
    strftime(o.order_date, '%Y-%m') AS month,
    i.product_id,
    SUM(i.quantity) AS units_sold,
    SUM(i.quantity * i.unit_list_price) AS gross_sales,
    SUM(i.discount_amount) AS discounts
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
WHERE o.status <> 'cancelled'
GROUP BY 1, 2
ORDER BY 1, 2
"""

run_sql(SALES_SQL, tiny)

,month,product_id,units_sold,gross_sales,discounts
0,2026-10,P001,1.0,40.0,0.0


The same answer on the test case. (Look closely: SQL wrote the units as `1.0` where pandas wrote `1`. The value is the same but the type is not. Keep that in mind for step 6.)

Now both versions on the full records: do they agree on all 24 rows?

In [6]:
from_pandas = sales_pandas(tables).sort_values(["month", "product_id"]).reset_index(drop=True)
from_sql = run_sql(SALES_SQL, tables)

print("Rows:", len(from_pandas), "and", len(from_sql))
difference = (from_pandas[["units_sold", "gross_sales", "discounts"]] - from_sql[["units_sold", "gross_sales", "discounts"]]).abs().max().max()
print("Largest difference in any number, to the cent:", round(difference, 2))

Rows: 24 and 24
Largest difference in any number, to the cent: 0.0


Identical. Two languages, one rule.

## Step 4: Where does the rule live?

Why would anyone write the same calculation twice? Because the right tool depends on how much data there is.

| Engine | Good for |
|---|---|
| **pandas** | Small data that fits in your computer's memory. Quick to write and test |
| **SQL** (a database) | Medium data. The database does the heavy lifting |
| **PySpark** | Very large data, spread across many machines |

A retailer starts small and grows. If the business rule is buried inside pandas code, moving to SQL means rewriting it and hoping nothing changed. So the design keeps the **rule** separate from the **engine** that runs it.

In [7]:
show_doc("HLD.md", "6. One rule, many engines")

> **From `docs/HLD.md`: 6. One rule, many engines**

| Idea | What it means here |
|---|---|
| Contract | A transformation is described without any engine in it: what goes in, what must come out, and the rule |
| Adapter | Each engine is a class with the same small interface. The pipeline talks to the interface and does not know which engine is behind it |
| Implementation per engine | The same transformation has a SQL query for the SQL engine and a function for the pandas engine |
| Standard output format | Every engine returns its result as an Apache Arrow table that matches the contract's columns and types. Arrow is a table format that pandas, SQL engines, and Spark all read and write |
| Conformance | The same hand-worked test cases run against every engine. All engines must agree, value for value |
| One switch | A setting chooses the engine. Nothing else changes |

| Engine | Suited to | In this milestone |
|---|---|---|
| pandas | Small data that fits in memory | Built and tested |
| SQL (DuckDB) | Medium data on one machine | Built and tested; the default |
| PySpark | Large data across many machines | Interface and file layout reserved; not built |

The rule lives in a **contract** (the code calls it a spec). It says what goes in, exactly what must come out, and the rule in words. There is no pandas and no SQL in it.

In [8]:
spec = SPECS["sales"]

print("Name:   ", spec.name)
print("Inputs: ", spec.inputs)
print("Output: ", [(column.name, column.type) for column in spec.output.columns])
print("Key:    ", spec.output.key, "(one row for each)")
print("Rule:   ", spec.rule)

Name:    sales
Inputs:  ('orders', 'order_items')
Output:  [('month', 'month'), ('product_id', 'string'), ('units_sold', 'integer'), ('gross_sales', 'decimal'), ('discounts', 'decimal')]
Key:     ('month', 'product_id') (one row for each)
Rule:    For order lines whose order is not cancelled, grouped by the month of the order's order_date and by product_id: units_sold is the sum of quantity; gross_sales is the sum of quantity x unit_list_price; discounts is the sum of discount_amount. A line that appears more than once is counted each time.


## Step 5: Engines are interchangeable

An **engine** is a small class that knows how to run one kind of code. Every engine has the same few methods, so the rest of the system can use any of them without caring which. Programmers call this the **adapter** pattern.

Here is the production `sales` transformation run on the SQL engine and then on the pandas engine. The only thing that changes is one word.

In [9]:
def run_on(engine_name, spec):
    engine = get_engine(engine_name)
    for name, table in load_sources().items():
        engine.register(name, table)
    return engine.run(spec)


by_sql = run_on("sql", spec)
by_pandas = run_on("pandas", spec)

print("Difference between the engines:", tables_differ(by_pandas, by_sql))
by_sql.to_pandas().head(3)

Difference between the engines: None


,month,product_id,units_sold,gross_sales,discounts
0,2026-09,P001,420,16800.0,840.0
1,2026-09,P002,260,9100.0,455.0
2,2026-09,P003,95,8550.0,855.0


`None` means no difference.

The third engine, PySpark, is not built in this milestone. But its place is ready: there is a class for it with the same methods, and it tells you plainly what it would need.

In [10]:
try:
    run_on("spark", spec)
except NotImplementedError as error:
    print(error)

The PySpark engine is not built yet. It is a reserved slot in the design. To build it: install Java and pyspark, implement register (create a Spark DataFrame from the Arrow table), load_implementation (import implementations/spark/<name>.py), and execute (call the function and return the result as an Arrow table), then run the conformance tests.


## Step 6: One standard format for results

pandas returns a DataFrame. A database returns rows. Spark returns something else again. If each engine handed back its own kind of table, everything downstream would need to know which engine ran.

You saw a small example in step 3, where one engine wrote `1.0` and the other `1`. So every engine returns its result in one standard format, an **Apache Arrow table**, and every result goes through `conform`, which checks it against the contract: the right columns, the right types, amounts rounded to cents, one row per key.

Watch what happens when a result breaks the contract. Here we "forget" the discounts column.

In [11]:
broken = by_sql.drop_columns(["discounts"])

try:
    conform(broken, spec.output)
except ContractError as error:
    print("Rejected:", error)

Rejected: sales: columns do not match the contract. Missing: ['discounts']. Unexpected: none.


This strictness matters more than it looks. In the next notebook a model will write the code. `conform` is what stops a model's almost-right table from slipping through.

## Step 7: A trap worth knowing: sum first, then join

Here is the most common mistake in this kind of work. Suppose you want sales and refunds side by side, and you join order lines straight to refunds.

Look at this rain jacket line. It has **two** refunds.

In [12]:
refund_counts = tables["refunds"].groupby("order_item_id").size()
twice = refund_counts[refund_counts > 1].index[0]

line = tables["order_items"][tables["order_items"]["order_item_id"] == twice]
display(line)
display(tables["refunds"][tables["refunds"]["order_item_id"] == twice])

,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount
1064,OI-01065,ORD-21034,P001,4,40.0,8.0


,refund_id,order_item_id,product_id,refund_date,refund_amount,return_handling_cost
21,REF-0022,OI-01065,P001,2026-09-30,38.0,6.0
31,REF-0032,OI-01065,P001,2026-10-05,36.0,6.0


In [13]:
joined = line.merge(tables["refunds"], on="order_item_id")

print("The line's real quantity:        ", line["quantity"].sum())
print("Quantity after joining to refunds:", joined["quantity"].sum())

The line's real quantity:         4
Quantity after joining to refunds: 8


The join repeated the line once for each refund, so its quantity is counted twice. Sales are now inflated, and nothing raised an error.

The fix is a rule the design follows everywhere: **sum each table on its own first, then join the sums.** That is why there are six small transformations and not one big query.

In [14]:
show_doc("HLD.md", "5. The transformations")

> **From `docs/HLD.md`: 5. The transformations**

Six transformations produce the monthly profit table. Each is small enough to reason about and test by hand.

*(Diagram: open `docs/HLD.md` to see it.)*

| Transformation | What it calculates |
|---|---|
| `sales` | Units sold, gross sales, and discounts, by product and order month, leaving out cancelled orders |
| `refund_totals` | Refunds and return handling cost, by product and refund month |
| `delivery_costs` | Pick-and-pack and carrier cost, by product and ship month |
| `fee_totals` | Payment fees, by product and capture month |
| `product_cost` | Units sold times unit cost. Unavailable when the cost is missing |
| `monthly_product_profit` | Brings the five together and calculates net sales, gross profit, and contribution profit |

## Step 8: The full set of test cases

The cancelled-order case you wrote is one of 22 hand-worked cases, covering all six transformations plus one extra task you'll meet in the next notebook. Each has its inputs written out in full and its output worked out by hand.

In [15]:
cases = load_cases()

pd.set_option("display.max_colwidth", None)
pd.DataFrame([{"transformation": case.transformation, "case": case.name, "why it is there": case.why} for case in cases])

,transformation,case,why it is there
0,sales,a plain order,"Two units at list price, no discount."
1,sales,a discounted line,"The discount is recorded, not taken off gross sales."
2,sales,a cancelled order is left out,Only the completed order counts.
3,sales,two months and two products,Sales are grouped by the order's month and by product.
4,sales,a line that appears twice is counted twice,"Removing duplicates is a quality check for milestone 3. Until then, count what is there."
5,refund_totals,one refund,
6,refund_totals,two refunds on one line,A partial refund followed by the rest. Both count.
7,refund_totals,a refund counts in the month it is issued,"The month comes from refund_date, whenever the order was placed."
8,delivery_costs,one shipment,
9,delivery_costs,several products and months,Costs count in the month the line shipped.


The production implementations must pass every case, on both engines. `run_cases` runs them.

In [16]:
rows = []
for name, task in TASKS.items():
    for engine_name in ["sql", "pandas"]:
        results = run_cases(task, engine_name)
        rows.append({"transformation": name, "engine": engine_name, "passed": f"{sum(result.passed for result in results)} of {len(results)}"})

pd.DataFrame(rows).pivot(index="transformation", columns="engine", values="passed")

engine,pandas,sql
transformation,,
delivery_costs,2 of 2,2 of 2
fee_totals,2 of 2,2 of 2
monthly_product_profit,4 of 4,4 of 4
product_cost,3 of 3,3 of 3
profit_all_in_one,3 of 3,3 of 3
refund_totals,3 of 3,3 of 3
sales,5 of 5,5 of 5


## What we now know

- **A hand-worked test case** is a small input with an output a person worked out. It is how you know a calculation is right.
- **The rule lives in a contract**, with no engine in it. Each engine has its own implementation of the rule.
- **Engines are interchangeable.** They share one interface and return one standard table format, and the same test cases judge all of them.
- **Sum first, then join.** Joining detail to detail quietly inflates totals.

You now have a yardstick: 22 cases that say what "correct" means. That is exactly what you need before letting a model write code for you.

**Next: notebook 3**, where a model drafts a transformation and the tests judge it.